# 实机测试：Kasli TTL → Spectrum EXT0 → 示波器 CH0

本测试只发 **1 个 TTL 上升沿**。卡在 CH0 播放 **50 µs 脉冲 → 25 µs 暗时间 → 50 µs 脉冲**；载波 200 kHz，第二段附加 90°。目的是看线是否接对、波形是否出现，不验证相对 Kasli 的绝对相位或模拟延迟。不需要校准 JSON。

**接线示例：Kasli 的一个 TTLOut → 一张 Spectrum AWG 的 EXT0；该卡 CH0 → 示波器。** 运行实机前按现场核对的接线填写设备库、TTL 和卡序列号；默认配置只用于离线编译。EXT0 取上升沿（默认阈值 1 V、50 Ω）；卡使用自己的内部 PLL，采样率 1.25 GS/s；不用接参考时钟线，TTL 只启动回放。此模式不保证与 Kasli 同频同相。CH0 → 示波器 50 Ω；如分接观察 TTL，使用高阻探头。先核对 DIO 驱动能力、EXT0 和其他输出接线。**独占整个 core 和整张卡**：kernel 会 `core.reset()`、清空待执行 RTIO 事件；卡会停止原回放并关闭未用输出。勿与其他实验、idle kernel 或 MonInj 强制输出争用。

选项目 `.venv` 内核。依次运行下列单元；编译和画图都不访问设备；只有最后一格会运行实机实验。

In [ ]:
import math
import os
import runpy
from pathlib import Path

from taiyidsl import (
    DeviceSetup,
    Experiment,
    LabConfig,
    Qubit,
    Square,
    delay,
    kHz,
    play,
    us,
)
from taiyidsl.config.descriptor import resolve_device

ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/taiyidsl").is_dir()
)
# 实机：填写可信设备库路径、卡铭牌序列号和接线对应的 TTL。
DEVICE_DB = None  # 例如 ROOT / "local_config/device_db.py"
CARD_SERIAL = "OFFLINE"
CARD_ADDRESS = None  # 实机默认按序列号找卡；也可填写现场确认的设备地址
TTL = "ttl20"  # 离线接线示例
ARTIQ_PYTHON = os.environ.get("TAIYIDSL_ARTIQ_PYTHON") or str(
    ROOT / "local_config/hardware-worker/bin/taiyidsl-artiq-python"
)
RANGE_MV = 500

In [ ]:
if DEVICE_DB is None:
    db = {
        "core": {
            "type": "local",
            "module": "artiq.coredevice.core",
            "class": "Core",
            "arguments": {"host": None, "ref_period": 1e-9, "target": "rv32g"},
        },
        "ttl20": {
            "type": "local",
            "module": "artiq.coredevice.ttl",
            "class": "TTLOut",
            "arguments": {"channel": 20},
        },
    }
else:
    db = runpy.run_path(str(DEVICE_DB))["device_db"]
trigger, ttl = resolve_device(db, TTL)
channel = ttl["arguments"]["channel"]
if (
    trigger != TTL
    or trigger != f"ttl{channel}"
    or ttl.get("module") != "artiq.coredevice.ttl"
    or ttl.get("class") != "TTLOut"
):
    raise ValueError("设备库中的 TTL 名、方向或 RTIO 号与所选卡的固定接线不符。")
if (
    "ttl_Orca_Cam" in db
    and resolve_device(db, "ttl_Orca_Cam")[1]["arguments"]["channel"] == channel
):
    raise ValueError("当前设备库已将此通道分配给 ORCA 相机。")

spec_db = {
    "A": {
        "serial_number": CARD_SERIAL,
        "clock_policy": "internal_pll_bench",  # 单卡、EXT0 触发、不使用外参考
        "address": CARD_ADDRESS,
        "active_channels": [0],
        "output_range_mv": RANGE_MV,
        "trigger_latency_samples": 64,
    }
}
# 64 是原测试留给编译器的延迟估计，不是实测值；示波器读数为准。
descriptor = {
    "backend": "artiq_spectrum",
    "instruments": {
        "core": {"kind": "artiq", "device": "core"},
        "A": {"kind": "spectrum", "device": "A"},
    },
    "channels": {"ch0": {"instrument": "A", "channel": 0}},
    "signals": {"q0": {"clock": "ch0"}},
    "sync_groups": {"clock_awg": {"cards": ["A"], "trigger": trigger}},
}
setup = DeviceSetup.from_descriptor(
    descriptor, device_db={"core": db["core"], trigger: ttl}, spec_db=spec_db
)
print(f"{trigger} (RTIO {channel:#x}) → EXT0；Spectrum S/N {CARD_SERIAL} CH0 → 示波器")

## 双脉冲：编译一次，不发射

In [ ]:
# domain specific language

q0 = Qubit("q0", signals={"clock": "q0.clock"})
lab = LabConfig(
    setup,
    qubits=[q0],
    emulation=DEVICE_DB is None or CARD_SERIAL == "OFFLINE",
    artiq_python=ARTIQ_PYTHON,
)
pulse = Square(duration=50 * us, amplitude=0.2, frequency=200 * kHz)
gap = 25 * us
second_phase = math.pi / 2

exp = Experiment("one_trigger_clock_ramsey", [q0])
with exp.average(1), exp.realtime(), exp.triggered_sequence(group="clock_awg"):
    play(q0.clock, pulse)
    delay(gap)
    play(q0.clock, pulse, phase=second_phase)

compiled = lab.compile(exp)
print("已编译 1 shot；尚未连接仪器。")

**可选：看一下待发的 DAC 波形。** 不改配置，不连接设备；也可以直接跳到最后一格。横轴从 AWG 回放起点算，不是 TTL 到模拟输出的实测延迟。

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from taiyidsl.backends.artiq_spectrum.awg import reconstruct_window

group = next(
    e.params for e in compiled.points[0].physical_events if e.kind == "spectrum_program"
)
program = group["cards"]["A"]["program"]
y = reconstruct_window(program, 0, program["physical_duration_samples"])[:, 0]
plt.figure(figsize=(8, 2.5))
plt.plot(np.arange(len(y)) * 1e6 / program["card_spec"]["sample_rate_hz"], y)
plt.xlabel("Time from AWG replay start (µs)")
plt.ylabel("CH0 DAC code")
plt.show()

## 运行实验：ARTIQ 自动触发 Spectrum

先按前面的配置选择可信的实机设备库、序列号和 TTL，并重新编译。启动环境见 [Linux worker 配置](../nix_hardware_worker.md)。确认整张卡和 core 可独占，卡已设为 **Internal PLL、1.25 GS/s**，把示波器设成上升沿单次采集，再把下一格改为 `RUN_EXPERIMENT = True` 并执行实验。`triggered_sequence` 已把 TTL 上升沿和下降沿编入 ARTIQ 实验。运行时，ARTIQ 先完成初始化；Spectrum 上传并等待 EXT0，卡就绪后 ARTIQ 自动执行该 shot，随后等待卡完成并释放。**不需要按触发键，也不用另写 TTL 命令。** 回执记录按设定值回放，不代表已测量模拟延迟、增益或绝对相位。

默认应看到一个 **64 ns** TTL，CH0 上两串各 **10 个周期**的波形，中间相隔 **25 µs**。出现异常时可能已经发过 TTL，先检查示波器，不要自动重试。

In [ ]:
RUN_EXPERIMENT = False  # 仅在现场确认就绪后手动改为 True，执行一次
if RUN_EXPERIMENT:
    with lab.connect() as session:
        print(session.observed["cards"]["A"])
        receipt = session.run(compiled)
    print(receipt["spectrum_receipts"][0]["receipt"]["cards"]["A"])

如提示时钟不符，请对照报错中的期望值和读回值。`connect()` 和 `run()` 不会自动改钟；若卡还在外参考模式，先在设备独占且停止回放时单独切换到内部 PLL，并释放卡。不是将外参考的频率改成 0。

要做与 ARTIQ 共参考的校准实验，应恢复默认时钟策略，核对现场参考源、锁定及校准后重新编译。两种模式都用同一个 `lab.connect()` / `session.run(compiled)`。